# 02. 벤치마크 조건 감사하기

## 목표
모델 점수만 보지 않고 하네스, 추론 노력, 반복 횟수와 출처를 함께 기록하는 방법을 연습합니다.

In [ ]:
from dataclasses import dataclass

@dataclass
class Evaluation:
    model: str
    harness: str
    effort: str
    runs: int | None
    source: str

evaluations = [
    Evaluation("Kimi K3", "KimiCode", "max", None, "vendor"),
    Evaluation("Comparison A", "Codex", "max", None, "external"),
    Evaluation("Comparison B", "Claude Code", "unknown", 3, "external"),
]
for item in evaluations:
    print(item)

In [ ]:
def comparability_warnings(rows: list[Evaluation]) -> list[str]:
    warnings = []
    # 비교군에 서로 다른 값이 있으면 모델 외 요인이 섞였음을 표시합니다.
    for field in ("harness", "effort", "runs", "source"):
        values = {getattr(row, field) for row in rows}
        if len(values) > 1:
            warnings.append(f"{field} 조건이 동일하지 않음: {sorted(map(str, values))}")
    return warnings

for warning in comparability_warnings(evaluations):
    print("WARNING:", warning)

## 모델 카드 대표 점수 점검
아래 값은 2026-07-28 공식 모델 카드의 제작사 보고값 일부입니다. 서로 다른 benchmark 점수를 평균내어 종합 성능으로 해석하지 않고, 어떤 영역에서 후속 독립 검증이 필요한지 분류합니다.

In [ ]:
reported_scores = {
    "GPQA Diamond": {"area": "reasoning", "score": 93.5},
    "Terminal-Bench 2.1": {"area": "coding", "score": 88.3},
    "BrowseComp": {"area": "agentic", "score": 91.2},
    "OmniDocBench": {"area": "vision", "score": 91.1},
}

for benchmark, item in reported_scores.items():
    print(f"{item['area']:>9} | {benchmark:<20} | {item['score']:>5.1f}")

# 점수마다 scale, harness, tool 사용, 반복 횟수가 다를 수 있습니다.
# 같은 숫자 범위라도 benchmark 사이의 직접 평균은 의미가 없습니다.

## 해석 질문
- 하네스가 다르면 도구 사용과 재시도 정책이 점수에 어떤 영향을 줄까요?
- 평균 점수만 있고 분산과 실행 횟수가 없다면 무엇을 알 수 없을까요?
- 제조사 내부 평가와 독립 평가를 표에서 어떻게 구분해야 할까요?
- `reasoning_effort=max`, temperature 1.0 조건을 다른 모델과 동일하게 맞췄는지 어떻게 감사할까요?

실제 평가표를 만들 때는 과제 버전, GPU, 시간·토큰 예산, tool augmentation, fallback 발생 여부도 열로 추가하세요.